In [1]:
using JuMP, HiGHS, DataFrames

In [2]:
df = DataFrame(
    :Assistant => [0, 1],
    :Task01 => [100, 80],
    :Task02 => [85, 70],
    :Task03 => [40, 90],
    :Task04 => [45, 85],
    :Task05 => [70, 80],
    :Task06 => [82, 65],
)

Row,Assistant,Task01,Task02,Task03,Task04,Task05,Task06
,Int64,Int64,Int64,Int64,Int64,Int64,Int64
1,0,100,85,40,45,70,82
2,1,80,70,90,85,80,65


In [3]:
assistants = df[:, :Assistant]

2-element Vector{Int64}:
 0
 1

In [4]:
tasks = names(df)[2:end]

6-element Vector{String}:
 "Task01"
 "Task02"
 "Task03"
 "Task04"
 "Task05"
 "Task06"

In [5]:
model = Model(HiGHS.Optimizer)

A JuMP Model
├ solver: HiGHS
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [6]:
@variable(model, x[assistants, tasks], Bin)

2-dimensional DenseAxisArray{VariableRef,2,...} with index sets:
    Dimension 1, [0, 1]
    Dimension 2, ["Task01", "Task02", "Task03", "Task04", "Task05", "Task06"]
And data, a 2×6 Matrix{VariableRef}:
 x[0,Task01]  x[0,Task02]  x[0,Task03]  x[0,Task04]  x[0,Task05]  x[0,Task06]
 x[1,Task01]  x[1,Task02]  x[1,Task03]  x[1,Task04]  x[1,Task05]  x[1,Task06]

In [7]:
@constraint(model, [t in tasks], sum(x[a, t] for a in assistants) == 1)

1-dimensional DenseAxisArray{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.EqualTo{Float64}}, ScalarShape},1,...} with index sets:
    Dimension 1, ["Task01", "Task02", "Task03", "Task04", "Task05", "Task06"]
And data, a 6-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.EqualTo{Float64}}, ScalarShape}}:
 x[0,Task01] + x[1,Task01] = 1
 x[0,Task02] + x[1,Task02] = 1
 x[0,Task03] + x[1,Task03] = 1
 x[0,Task04] + x[1,Task04] = 1
 x[0,Task05] + x[1,Task05] = 1
 x[0,Task06] + x[1,Task06] = 1

In [8]:
@constraint(model, [a in assistants], sum(x[a, t] for t in tasks) == 3)

1-dimensional DenseAxisArray{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.EqualTo{Float64}}, ScalarShape},1,...} with index sets:
    Dimension 1, [0, 1]
And data, a 2-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.EqualTo{Float64}}, ScalarShape}}:
 x[0,Task01] + x[0,Task02] + x[0,Task03] + x[0,Task04] + x[0,Task05] + x[0,Task06] = 3
 x[1,Task01] + x[1,Task02] + x[1,Task03] + x[1,Task04] + x[1,Task05] + x[1,Task06] = 3

In [9]:
@constraint(model, [a in assistants], x[a,"Task05"] == x[a,"Task06"])

1-dimensional DenseAxisArray{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.EqualTo{Float64}}, ScalarShape},1,...} with index sets:
    Dimension 1, [0, 1]
And data, a 2-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.EqualTo{Float64}}, ScalarShape}}:
 x[0,Task05] - x[0,Task06] = 0
 x[1,Task05] - x[1,Task06] = 0

In [10]:
@objective(model, Max, sum(df[df.:Assistant .== a, t][1] * x[a, t] for a in assistants, t in tasks))

100 x[0,Task01] + 85 x[0,Task02] + 40 x[0,Task03] + 45 x[0,Task04] + 70 x[0,Task05] + 82 x[0,Task06] + 80 x[1,Task01] + 70 x[1,Task02] + 90 x[1,Task03] + 85 x[1,Task04] + 80 x[1,Task05] + 65 x[1,Task06]

In [11]:
optimize!(model)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
MIP has 10 rows; 12 cols; 28 nonzeros; 12 integer variables (12 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [4e+01, 1e+02]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 3e+00]
Presolving model
2 rows, 5 cols, 10 nonzeros 0s
1 rows, 5 cols, 5 nonzeros 0s
Presolve reductions: rows 1(-9); columns 5(-7); nonzeros 5(-23) 
Objective function is integral with scale 1

Solving MIP model with:
   1 row
   5 cols (5 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   5 nonzeros
   Thread count 4 (of 8 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate

In [12]:
if termination_status(model) == OPTIMAL
    println("Optimal solution found")
    for a in assistants
        for t in tasks
            if value(x[a, t]) > 0.5
                println("Assistant ", a, " is assigned to task ", t)
            end
        end
    end
else
    println("No optimal solution found")
end

Optimal solution found
Assistant 0 is assigned to task Task01
Assistant 0 is assigned to task Task05
Assistant 0 is assigned to task Task06
Assistant 1 is assigned to task Task02
Assistant 1 is assigned to task Task03
Assistant 1 is assigned to task Task04
